# AFSK SMS Receiver, 131072-sample capture

Step 1 loads the overlay and defines parameters. Step 2 checks ADC waveform. Step 3 captures and decodes one AFSK SMS frame.


In [ ]:
# Step 1: load overlay and configure all parameters.
from time import time
import os

import numpy as np
import matplotlib.pyplot as plt
from pynq import Overlay, allocate

BITFILE = 'base_add.bit'
CHANNEL = 'ch0'          # 'ch0' or 'ch1'
LOCAL_ADDR = None        # None accepts every address; or set 0x00..0x07

PL_CLK_HZ = 125_000_000
ADC_HALF_PERIOD = 3125
FS_HZ = PL_CLK_HZ / (2 * ADC_HALF_PERIOD)   # 20 kSPS
BIT_RATE = 100.0
BIT_SAMPLES = int(round(FS_HZ / BIT_RATE))  # 200 samples/bit, 10 ms/bit

MAX_SAMPLE_N = 262144
AFSK_SAMPLE_COUNT = 131072                  # 2x old 65536 window, 6.5536 s at 20 kSPS
ADC_TEST_SAMPLE_COUNT = 4096

SPACE_HZ = 1200.0
MARK_HZ = 2200.0
CENTER_FRACTION = 0.60
FAST_OFFSET_STRIDE = 10
MIN_ALT_BITS = 24
INVERT_BITS = True                          # This board/test path currently decodes correctly with inversion.
MAX_ATTEMPTS = 60

CTRL = 0x00
STATUS = 0x04
SAMPLE_COUNT_REG = 0x08
ADC_HALF = 0x0C
SAMPLE_DELAY = 0x10
DECIMATION = 0x14
CHANNEL_MASK = 0x18
CAPTURE_MODE = 0x1C
TRIGGER_MODE = 0x20
PRE_DELAY = 0x24
BUFFER_SELECT = 0x28
LATEST_A = 0x2C
LATEST_B = 0x30
SAMPLE_COUNTER = 0x34
FIFO_LEVEL = 0x38
ERROR_FLAGS = 0x3C
VERSION = 0x44
SAVED_COUNTER = 0x48
LAST_AXIS_WORD = 0x4C
DEBUG_STATE = 0x50
AXIS_SENT_COUNT = 0x54
AXIS_STALL_COUNT = 0x58
TLAST_COUNT = 0x5C
FIFO_BACKPRESSURE = 0x60
DROPPED_SAMPLE_COUNT = 0x64
CAPTURE_DONE_LATCHED = 0x68
CORE_DONE = 0x6C
S2MM_DMASR = 0x34
SENTINEL = np.uint32(0xDEADBEEF)


def find_overlay_attr(overlay, text):
    matches = [name for name in overlay.ip_dict.keys() if text in name.lower()]
    if not matches:
        raise RuntimeError('Cannot find IP containing %r. IPs: %r' % (text, list(overlay.ip_dict.keys())))
    name = matches[0]
    if '/' in name:
        raise RuntimeError('IP %r is hierarchical; bind it manually.' % name)
    return name, getattr(overlay, name)


def configure_capture(ctrl, sample_count, adc_half_period=ADC_HALF_PERIOD, capture_mode=1, decimation=1, sample_delay=1):
    ctrl.write(CTRL, 0x04)
    ctrl.write(CTRL, 0x00)
    ctrl.write(ERROR_FLAGS, 0xFFFFFFFF)
    ctrl.write(SAMPLE_COUNT_REG, int(sample_count))
    ctrl.write(ADC_HALF, int(adc_half_period))
    ctrl.write(SAMPLE_DELAY, int(sample_delay))
    ctrl.write(DECIMATION, int(decimation))
    ctrl.write(CHANNEL_MASK, 0b11)
    ctrl.write(CAPTURE_MODE, int(capture_mode))
    ctrl.write(TRIGGER_MODE, 0)
    ctrl.write(PRE_DELAY, 0)
    ctrl.write(BUFFER_SELECT, 0)


def capture_once(sample_count, capture_mode=1, adc_half_period=ADC_HALF_PERIOD, decimation=1, sample_delay=1, verbose=False):
    sample_count = min(max(int(sample_count), 1), MAX_SAMPLE_N)
    buf = allocate(shape=(sample_count,), dtype=np.uint32)
    buf[:] = SENTINEL
    buf.flush()

    dma.recvchannel.transfer(buf)
    configure_capture(ctrl, sample_count, adc_half_period, capture_mode, decimation, sample_delay)
    ctrl.write(CTRL, 0x01)
    ctrl.write(CTRL, 0x03)
    ctrl.write(CTRL, 0x01)

    t0 = time()
    dma.recvchannel.wait()
    elapsed = time() - t0
    buf.invalidate()

    raw = np.array(buf, dtype=np.uint32)
    ch0 = (raw & np.uint32(0x0FFF)).astype(np.float64)
    ch1 = ((raw >> np.uint32(16)) & np.uint32(0x0FFF)).astype(np.float64)

    axis_sent = ctrl.read(AXIS_SENT_COUNT)
    tlast = ctrl.read(TLAST_COUNT)
    dropped = ctrl.read(DROPPED_SAMPLE_COUNT)
    status = ctrl.read(STATUS)

    if verbose:
        print('DMA wait elapsed = %.6f s' % elapsed)
        print('sample_count     = %d' % sample_count)
        print('expected time    = %.6f s' % (sample_count / FS_HZ))
        print('AXIS_SENT_COUNT  = %d' % axis_sent)
        print('TLAST_COUNT      = %d' % tlast)
        print('DROPPED_SAMPLES  = %d' % dropped)
        print('STATUS           = 0x%08X' % status)
        print('ERROR_FLAGS      = 0x%08X' % ctrl.read(ERROR_FLAGS))

    sentinel_idx = np.flatnonzero(raw == SENTINEL)
    if len(sentinel_idx) != 0:
        raise RuntimeError('DMA buffer still contains sentinel values: first=%d, count=%d, axis_sent=%d, tlast=%d, status=0x%08X' % (sentinel_idx[0], len(sentinel_idx), axis_sent, tlast, status))
    if axis_sent != sample_count:
        raise RuntimeError('AXIS_SENT_COUNT mismatch: %d != %d' % (axis_sent, sample_count))
    if tlast != 1:
        raise RuntimeError('TLAST_COUNT is not 1: %d' % tlast)
    if dropped != 0:
        raise RuntimeError('Dropped samples detected: %d' % dropped)
    if (status & (1 << 10)) != 0:
        raise RuntimeError('STATUS.fatal_error is set.')

    return raw, ch0, ch1, elapsed


def select_channel(ch0, ch1):
    return ch1 if CHANNEL.lower() == 'ch1' else ch0


def tone_power(samples, freq_hz):
    x = samples.astype(np.float64)
    x = x - np.mean(x)
    n = np.arange(len(x), dtype=np.float64)
    phase = 2.0 * np.pi * freq_hz * n / FS_HZ
    i = np.sum(x * np.cos(phase))
    q = np.sum(x * np.sin(phase))
    return i * i + q * q


def crc8_0x07(data):
    crc = 0
    for byte in data:
        crc ^= byte & 0xFF
        for _ in range(8):
            if crc & 0x80:
                crc = ((crc << 1) ^ 0x07) & 0xFF
            else:
                crc = (crc << 1) & 0xFF
    return crc


def bytes_to_ascii(values):
    return ''.join(chr(v) if 32 <= v <= 126 else '.' for v in values)


def bits_to_bytes_lsb(bits, phase):
    out = []
    for start in range(phase, len(bits) - 7, 8):
        value = 0
        for i in range(8):
            value |= (bits[start + i] & 1) << i
        out.append(value)
    return out


def find_frame(byte_values):
    sync = [0xAA, 0xAA, 0xAA, 0xAA, 0x7E]
    for i in range(0, len(byte_values) - len(sync) - 3):
        if byte_values[i:i + len(sync)] != sync:
            continue
        base = i + len(sync)
        addr = byte_values[base]
        length = byte_values[base + 1]
        end = base + 2 + length + 1
        addr_ok = (LOCAL_ADDR is None) or (addr == LOCAL_ADDR) or (addr == 0xFF)
        if end > len(byte_values):
            return {
                'status': 'incomplete',
                'byte_index': i,
                'addr': addr,
                'length': length,
                'payload': [],
                'rx_crc': None,
                'calc_crc': None,
                'crc_ok': False,
                'addr_ok': addr_ok,
            }
        payload = byte_values[base + 2:base + 2 + length]
        rx_crc = byte_values[base + 2 + length]
        calc_crc = crc8_0x07([addr, length] + payload)
        return {
            'status': 'complete',
            'byte_index': i,
            'addr': addr,
            'length': length,
            'payload': payload,
            'rx_crc': rx_crc,
            'calc_crc': calc_crc,
            'crc_ok': rx_crc == calc_crc,
            'addr_ok': addr_ok,
        }
    return None


def demod_bits(samples, offset):
    x = samples.astype(np.float64)
    x = x - np.mean(x)

    margin = (1.0 - CENTER_FRACTION) * 0.5
    center_start = int(round(BIT_SAMPLES * margin))
    center_stop = int(round(BIT_SAMPLES * (1.0 - margin)))
    win_n = center_stop - center_start

    n = np.arange(win_n, dtype=np.float64)
    c1200 = np.cos(2.0 * np.pi * SPACE_HZ * n / FS_HZ)
    s1200 = np.sin(2.0 * np.pi * SPACE_HZ * n / FS_HZ)
    c2200 = np.cos(2.0 * np.pi * MARK_HZ * n / FS_HZ)
    s2200 = np.sin(2.0 * np.pi * MARK_HZ * n / FS_HZ)

    bits = []
    confs = []
    bit_count = (len(x) - offset) // BIT_SAMPLES
    for k in range(bit_count):
        a = offset + k * BIT_SAMPLES + center_start
        b = offset + k * BIT_SAMPLES + center_stop
        seg = x[a:b]
        if len(seg) != win_n:
            break
        seg = seg - np.mean(seg)
        e1200 = np.dot(seg, c1200) ** 2 + np.dot(seg, s1200) ** 2
        e2200 = np.dot(seg, c2200) ** 2 + np.dot(seg, s2200) ** 2
        bit = 1 if e2200 > e1200 else 0
        if INVERT_BITS:
            bit ^= 1
        conf = abs(e2200 - e1200) / max(e2200 + e1200, 1.0)
        bits.append(bit)
        confs.append(conf)
    return bits, confs


def best_0101(bits, confs):
    best = {'score': -1.0, 'start': 0, 'length': 0, 'polarity': 0}
    for polarity in (0, 1):
        run_start = 0
        run_len = 0
        run_conf = []
        for i, bit in enumerate(bits):
            expected = (i + polarity) & 1
            if bit == expected:
                if run_len == 0:
                    run_start = i
                    run_conf = []
                run_len += 1
                run_conf.append(confs[i] if i < len(confs) else 0.0)
            else:
                if run_len >= MIN_ALT_BITS:
                    score = run_len + float(np.mean(run_conf))
                    if score > best['score']:
                        best = {'score': score, 'start': run_start, 'length': run_len, 'polarity': polarity}
                run_len = 0
                run_conf = []
        if run_len >= MIN_ALT_BITS:
            score = run_len + float(np.mean(run_conf))
            if score > best['score']:
                best = {'score': score, 'start': run_start, 'length': run_len, 'polarity': polarity}
    return best


def search_frame(samples):
    best = {
        'frame': None,
        'score': -1.0,
        'offset': 0,
        'phase': 0,
        'preamble_bit': 0,
        'preamble_len': 0,
        'bits': [],
        'bytes': [],
    }
    for offset in range(0, BIT_SAMPLES, FAST_OFFSET_STRIDE):
        bits, confs = demod_bits(samples, offset)
        alt = best_0101(bits, confs)
        if alt['score'] > best['score']:
            best.update({
                'score': alt['score'],
                'offset': offset,
                'phase': alt['start'] % 8,
                'preamble_bit': alt['start'],
                'preamble_len': alt['length'],
                'bits': bits,
                'bytes': bits_to_bytes_lsb(bits, alt['start'] % 8),
            })
        for phase in range(8):
            byte_values = bits_to_bytes_lsb(bits, phase)
            frame = find_frame(byte_values)
            if frame is not None:
                return {
                    'frame': frame,
                    'score': alt['score'],
                    'offset': offset,
                    'phase': phase,
                    'preamble_bit': alt['start'],
                    'preamble_len': alt['length'],
                    'bits': bits,
                    'bytes': byte_values,
                }
    return best


def print_frame(frame):
    payload = frame['payload'] if frame['payload'] is not None else []
    print('\nAFSK SMS frame detected')
    print('status      =', frame['status'])
    print('addr        = 0x%02X' % frame['addr'])
    print('addr_ok     =', frame['addr_ok'])
    print('length      =', frame['length'])
    print('payload     =', bytes_to_ascii(payload))
    print('payload_hex =', ' '.join('%02X' % b for b in payload))
    if frame['rx_crc'] is None:
        print('crc rx/calc = incomplete')
    else:
        print('crc rx/calc = 0x%02X / 0x%02X' % (frame['rx_crc'], frame['calc_crc']))
    print('crc_ok      =', frame['crc_ok'])


overlay = Overlay(BITFILE)
ctrl_name, ctrl = find_overlay_attr(overlay, 'adc_capture')
dma_name, dma = find_overlay_attr(overlay, 'dma')

print('Loaded overlay:', BITFILE)
print('Capture IP    :', ctrl_name)
print('DMA IP        :', dma_name)
print('FS_HZ         : %.1f Hz' % FS_HZ)
print('BIT_SAMPLES   : %d samples/bit' % BIT_SAMPLES)
print('MAX_SAMPLE_N  : %d samples, %.3f s' % (MAX_SAMPLE_N, MAX_SAMPLE_N / FS_HZ))
print('AFSK capture  : %d samples, %.3f s' % (AFSK_SAMPLE_COUNT, AFSK_SAMPLE_COUNT / FS_HZ))
print('CHANNEL       :', CHANNEL)
print('INVERT_BITS   :', INVERT_BITS)


In [ ]:
# Step 2: capture a short ADC segment and display 4 cycles of the selected test tone.
# Set TEST_TONE_HZ to 1200.0 or 2200.0 according to the signal you are feeding now.
TEST_TONE_HZ = 1200.0

raw, ch0, ch1, elapsed = capture_once(ADC_TEST_SAMPLE_COUNT, capture_mode=1, verbose=True)
samples = select_channel(ch0, ch1)
samples_dc = samples - np.mean(samples)

window_n = int(round(FS_HZ * 4.0 / TEST_TONE_HZ))
window_n = min(window_n, len(samples_dc))
seg = samples_dc[:window_n]
t_ms = np.arange(window_n) / FS_HZ * 1000.0

p1200 = tone_power(seg, SPACE_HZ)
p2200 = tone_power(seg, MARK_HZ)
measured_hz = MARK_HZ if p2200 > p1200 else SPACE_HZ
raw_bit = 1 if p2200 > p1200 else 0
bit = raw_bit ^ (1 if INVERT_BITS else 0)
confidence = abs(p2200 - p1200) / max(p2200 + p1200, 1.0)

print('channel      =', CHANNEL)
print('window len   = %d samples, %.3f ms' % (window_n, window_n / FS_HZ * 1000.0))
print('vpp          = %.1f ADC counts' % (float(np.max(seg) - np.min(seg))))
print('E1200/E2200  = %.3e / %.3e' % (p1200, p2200))
print('measured_hz  = %.0f Hz' % measured_hz)
print('decoded bit  = %d  (after INVERT_BITS=%s)' % (bit, INVERT_BITS))
print('confidence   = %.3f' % confidence)

plt.figure(figsize=(12, 4))
plt.plot(t_ms, seg, marker='o', linewidth=1.5)
plt.title('%s %.0f Hz check, 4 cycles' % (CHANNEL.upper(), TEST_TONE_HZ))
plt.xlabel('Time (ms)')
plt.ylabel('ADC counts, DC removed')
plt.grid(True)
plt.show()


In [ ]:
# Step 3: repeatedly capture 131072 samples, decode one AFSK SMS frame, then stop this cell.
for attempt in range(1, MAX_ATTEMPTS + 1):
    print('attempt %02d: capturing...' % attempt, flush=True)
    t0 = time()
    raw, ch0, ch1, capture_elapsed = capture_once(AFSK_SAMPLE_COUNT, capture_mode=1, verbose=False)

    print('attempt %02d: captured, decoding...' % attempt, flush=True)
    samples = select_channel(ch0, ch1)
    result = search_frame(samples)
    elapsed = time() - t0

    preamble_ms = (result['offset'] + result['preamble_bit'] * BIT_SAMPLES) / FS_HZ * 1000.0
    print('attempt %02d: %.3f s, score=%.3f, offset=%d, phase=%d, preamble=%.1f ms, len=%d bits' % (
        attempt,
        elapsed,
        result['score'],
        result['offset'],
        result['phase'],
        preamble_ms,
        result['preamble_len'],
    ))

    frame = result['frame']
    if frame is None:
        continue

    print_frame(frame)
    break
else:
    print('No complete frame detected after %d attempts.' % MAX_ATTEMPTS)
